# 27 clean liana global labels

**Purpose:** Standardize LIANA labels and focused biological axes.

**Original analysis notebook:** `24b_fix_LIANA_global_main_labels_and_axes_HVG8000_r0_6_v2_fixed_source_clean.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

# 43b — Fix LIANA global main labels and biological axes

This notebook **does not rerun LIANA**. It post-processes the completed `43_run_LIANA_global_main` outputs.

Reason: the LIANA run itself completed correctly, but non-malignant high-confidence labels were saved with the prefix `Malignant_non_malignant_*`. That makes the biological axis summaries misleading because the helper function interpreted every label beginning with `Malignant_` as malignant/cycling.

This notebook:
- reads the existing LIANA result tables,
- cleans non-malignant labels to `NonMalignant_*`,
- recalculates source/target broad groups,
- rebuilds source-target, axis, biological-axis and LR summaries,
- saves clean-label outputs with suffix `_clean_labels`.


In [ ]:
# Imports
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)


In [ ]:
# Paths
import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
METADATA_DIR = PROJECT_DIR / "metadata"
FIG_DIR = PROJECT_DIR / "figures" / "LIANA_HVG8000_r0_6"
FIG_DIR.mkdir(parents=True, exist_ok=True)

OLD_PREFIX = "LIANA_global_main_HVG8000_r0_6"
OUT_PREFIX = "LIANA_global_main_HVG8000_r0_6_clean_labels"

RANK_SIG_PATH = METADATA_DIR / f"{OLD_PREFIX}_rank_threshold_or_top_interactions.tsv"
TOP_GLOBAL_PATH = METADATA_DIR / f"{OLD_PREFIX}_top_global_interactions.tsv"
TOP_PER_PAIR_PATH = METADATA_DIR / f"{OLD_PREFIX}_top_source_target_interactions.tsv"
FULL_RESULTS_PATH = METADATA_DIR / f"{OLD_PREFIX}_all_results.tsv.gz"
COUNTS_PATH = METADATA_DIR / f"{OLD_PREFIX}_celltype_counts_used.tsv"
RUN_SUMMARY_PATH = METADATA_DIR / f"{OLD_PREFIX}_run_summary.tsv"

print("RANK_SIG_PATH:", RANK_SIG_PATH)
print("FULL_RESULTS_PATH:", FULL_RESULTS_PATH)


In [ ]:
# Helper functions
def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="\t", index=False)
    print("[WRITE]", path, df.shape)
    return path

def savefig(path, dpi=180):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()
    print("[FIG]", path)
    return path

def clean_liana_label(x):
    x = str(x)
    # Fix the specific naming issue from the original LIANA preparation:
    # high-confidence non-malignant populations were accidentally prefixed with "Malignant_non_malignant_".
    if x.startswith("Malignant_non_malignant_"):
        return "NonMalignant_" + x[len("Malignant_non_malignant_"):]
    return x

def broad_group_clean(x):
    x = clean_liana_label(x)
    if x.startswith("Malignant_"):
        return "malignant_or_cycling"
    if x.startswith("NonMalignant_"):
        return "non_malignant_reference_like"
    return "other"

def lineage_simplified(x):
    t = clean_liana_label(x).lower()
    if t.startswith("malignant_"):
        return "malignant_state"
    if "endothelial" in t:
        return "endothelial"
    if "mural" in t or "pericyte" in t or "fibroblast" in t:
        return "mural_pericyte_fibroblast"
    if "oligodendrocyte" in t:
        return "oligodendrocyte_reference"
    if "b_plasma" in t or "plasma" in t:
        return "B_plasma"
    return "other"

def biological_axis_clean(row):
    # Robust to either already-cleaned columns or broad-group columns.
    # Earlier v1 expected source_clean/target_clean but apply_clean_annotations
    # only created source/target; this version avoids that KeyError.
    if "source_broad_group" in row.index and "target_broad_group" in row.index:
        src = row["source_broad_group"]
        tgt = row["target_broad_group"]
    else:
        src_label = row["source_clean"] if "source_clean" in row.index else row.get("source", "")
        tgt_label = row["target_clean"] if "target_clean" in row.index else row.get("target", "")
        src = broad_group_clean(src_label)
        tgt = broad_group_clean(tgt_label)

    if src == "malignant_or_cycling" and tgt == "non_malignant_reference_like":
        return "malignant_to_microenvironment"
    if src == "non_malignant_reference_like" and tgt == "malignant_or_cycling":
        return "microenvironment_to_malignant"
    if src == "malignant_or_cycling" and tgt == "malignant_or_cycling":
        return "malignant_state_to_malignant_state"
    if src == "non_malignant_reference_like" and tgt == "non_malignant_reference_like":
        return "microenvironment_to_microenvironment"
    return "other"


In [ ]:
# Read original outputs and readiness
readiness_records = []
for label, path in [
    ("rank_threshold_or_top_interactions", RANK_SIG_PATH),
    ("top_global_interactions", TOP_GLOBAL_PATH),
    ("top_source_target_interactions", TOP_PER_PAIR_PATH),
    ("all_results", FULL_RESULTS_PATH),
    ("celltype_counts_used", COUNTS_PATH),
    ("run_summary", RUN_SUMMARY_PATH),
]:
    readiness_records.append({
        "item": label,
        "path": str(path),
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / 1024 / 1024, 3) if path.exists() else np.nan,
    })

readiness = pd.DataFrame(readiness_records)
write_tsv_replace(readiness, METADATA_DIR / f"{OUT_PREFIX}_readiness.tsv")
display(readiness)

if not RANK_SIG_PATH.exists():
    raise FileNotFoundError(RANK_SIG_PATH)

rank_sig = pd.read_csv(RANK_SIG_PATH, sep="\t")
top_global = pd.read_csv(TOP_GLOBAL_PATH, sep="\t") if TOP_GLOBAL_PATH.exists() else rank_sig.head(1000).copy()
top_per_pair = pd.read_csv(TOP_PER_PAIR_PATH, sep="\t") if TOP_PER_PAIR_PATH.exists() else None

print("[rank_sig]", rank_sig.shape)
display(rank_sig.head())


In [ ]:
# Apply clean labels and reclassify axes
def apply_clean_annotations(df):
    df = df.copy()

    # Keep original LIANA labels for traceability.
    df["source_original"] = df["source"].astype(str)
    df["target_original"] = df["target"].astype(str)

    # Create explicit clean labels. Keep source/target as clean labels too,
    # because downstream summaries expect source and target.
    df["source_clean"] = df["source_original"].map(clean_liana_label)
    df["target_clean"] = df["target_original"].map(clean_liana_label)
    df["source"] = df["source_clean"]
    df["target"] = df["target_clean"]

    # Broad biological groups and axes.
    df["source_broad_group"] = df["source_clean"].map(broad_group_clean)
    df["target_broad_group"] = df["target_clean"].map(broad_group_clean)
    df["communication_axis"] = df["source_broad_group"] + "_to_" + df["target_broad_group"]
    df["biological_axis"] = df.apply(biological_axis_clean, axis=1)
    df["source_lineage_simplified"] = df["source_clean"].map(lineage_simplified)
    df["target_lineage_simplified"] = df["target_clean"].map(lineage_simplified)
    return df

rank_sig_clean = apply_clean_annotations(rank_sig)
top_global_clean = apply_clean_annotations(top_global)

# Save clean interaction tables.
write_tsv_replace(rank_sig_clean, METADATA_DIR / f"{OUT_PREFIX}_rank_threshold_or_top_interactions.tsv")
write_tsv_replace(top_global_clean, METADATA_DIR / f"{OUT_PREFIX}_top_global_interactions.tsv")

if top_per_pair is not None:
    top_per_pair_clean = apply_clean_annotations(top_per_pair)
    write_tsv_replace(top_per_pair_clean, METADATA_DIR / f"{OUT_PREFIX}_top_source_target_interactions.tsv")

# Clean celltype counts.
if COUNTS_PATH.exists():
    counts = pd.read_csv(COUNTS_PATH, sep="\t")
    counts["liana_celltype_original"] = counts["liana_celltype"].astype(str)
    counts["liana_celltype"] = counts["liana_celltype_original"].map(clean_liana_label)
    counts["broad_group"] = counts["liana_celltype"].map(broad_group_clean)
    counts["lineage_simplified"] = counts["liana_celltype"].map(lineage_simplified)
    write_tsv_replace(counts, METADATA_DIR / f"{OUT_PREFIX}_celltype_counts_used.tsv")
    display(counts)


In [ ]:
# Recalculate source-target summary
source_target_summary = (
    rank_sig_clean
    .groupby(["source", "target", "source_broad_group", "target_broad_group", "communication_axis", "biological_axis"], observed=True)
    .agg(
        n_prioritized_interactions=("lr_pair", "nunique"),
        median_primary_rank=("primary_rank", "median"),
        min_primary_rank=("primary_rank", "min"),
    )
    .reset_index()
    .sort_values(["n_prioritized_interactions", "median_primary_rank"], ascending=[False, True])
)
write_tsv_replace(source_target_summary, METADATA_DIR / f"{OUT_PREFIX}_source_target_summary.tsv")
display(source_target_summary.head(30))

axis_summary = (
    rank_sig_clean
    .groupby("communication_axis", observed=True)
    .agg(
        n_prioritized_interactions=("lr_pair", "nunique"),
        n_source_target_pairs=("source", lambda x: len(set(zip(rank_sig_clean.loc[x.index, "source"], rank_sig_clean.loc[x.index, "target"])))),
        median_primary_rank=("primary_rank", "median"),
        min_primary_rank=("primary_rank", "min"),
    )
    .reset_index()
    .sort_values(["n_prioritized_interactions", "median_primary_rank"], ascending=[False, True])
)
write_tsv_replace(axis_summary, METADATA_DIR / f"{OUT_PREFIX}_axis_summary.tsv")
display(axis_summary)


In [ ]:
# Recalculate biological axis and ligand-receptor summaries
bio_summary = (
    rank_sig_clean
    .groupby(["biological_axis", "source_lineage_simplified", "target_lineage_simplified"], observed=True)
    .agg(
        n_interactions=("lr_pair", "nunique"),
        n_source_target_pairs=("source", lambda x: len(set(zip(rank_sig_clean.loc[x.index, "source"], rank_sig_clean.loc[x.index, "target"])))),
        median_primary_rank=("primary_rank", "median"),
        min_primary_rank=("primary_rank", "min"),
    )
    .reset_index()
    .sort_values(["n_interactions", "median_primary_rank"], ascending=[False, True])
)
write_tsv_replace(bio_summary, METADATA_DIR / f"{OUT_PREFIX}_biological_axis_summary.tsv")
display(bio_summary.head(50))

lr_summary = (
    rank_sig_clean
    .groupby(["ligand_complex", "receptor_complex", "lr_pair"], observed=True)
    .agg(
        n_source_target_pairs=("source", lambda x: len(set(zip(rank_sig_clean.loc[x.index, "source"], rank_sig_clean.loc[x.index, "target"])))),
        n_sources=("source", "nunique"),
        n_targets=("target", "nunique"),
        median_primary_rank=("primary_rank", "median"),
        min_primary_rank=("primary_rank", "min"),
    )
    .reset_index()
    .sort_values(["n_source_target_pairs", "median_primary_rank"], ascending=[False, True])
)
write_tsv_replace(lr_summary, METADATA_DIR / f"{OUT_PREFIX}_ligand_receptor_summary.tsv")
display(lr_summary.head(30))


In [ ]:
# Focused malignant-microenvironment interaction summaries
# Useful for deciding which axes to validate with Squidpy/OmniPath later.
mal_to_micro = rank_sig_clean[rank_sig_clean["biological_axis"].eq("malignant_to_microenvironment")].copy()
micro_to_mal = rank_sig_clean[rank_sig_clean["biological_axis"].eq("microenvironment_to_malignant")].copy()
mal_to_mal = rank_sig_clean[rank_sig_clean["biological_axis"].eq("malignant_state_to_malignant_state")].copy()

write_tsv_replace(mal_to_micro, METADATA_DIR / f"{OUT_PREFIX}_malignant_to_microenvironment_interactions.tsv")
write_tsv_replace(micro_to_mal, METADATA_DIR / f"{OUT_PREFIX}_microenvironment_to_malignant_interactions.tsv")
write_tsv_replace(mal_to_mal, METADATA_DIR / f"{OUT_PREFIX}_malignant_state_to_malignant_state_interactions.tsv")

focused_summary = pd.DataFrame([
    {
        "axis": "malignant_to_microenvironment",
        "n_rows": len(mal_to_micro),
        "n_unique_lr_pairs": mal_to_micro["lr_pair"].nunique(),
        "n_source_target_pairs": len(set(zip(mal_to_micro["source"], mal_to_micro["target"]))),
        "median_primary_rank": mal_to_micro["primary_rank"].median() if len(mal_to_micro) else np.nan,
        "min_primary_rank": mal_to_micro["primary_rank"].min() if len(mal_to_micro) else np.nan,
    },
    {
        "axis": "microenvironment_to_malignant",
        "n_rows": len(micro_to_mal),
        "n_unique_lr_pairs": micro_to_mal["lr_pair"].nunique(),
        "n_source_target_pairs": len(set(zip(micro_to_mal["source"], micro_to_mal["target"]))),
        "median_primary_rank": micro_to_mal["primary_rank"].median() if len(micro_to_mal) else np.nan,
        "min_primary_rank": micro_to_mal["primary_rank"].min() if len(micro_to_mal) else np.nan,
    },
    {
        "axis": "malignant_state_to_malignant_state",
        "n_rows": len(mal_to_mal),
        "n_unique_lr_pairs": mal_to_mal["lr_pair"].nunique(),
        "n_source_target_pairs": len(set(zip(mal_to_mal["source"], mal_to_mal["target"]))),
        "median_primary_rank": mal_to_mal["primary_rank"].median() if len(mal_to_mal) else np.nan,
        "min_primary_rank": mal_to_mal["primary_rank"].min() if len(mal_to_mal) else np.nan,
    },
])
write_tsv_replace(focused_summary, METADATA_DIR / f"{OUT_PREFIX}_focused_axis_summary.tsv")
display(focused_summary)


In [ ]:
# Figures with clean labels
figure_records = []

# Source-target count heatmap
pivot_counts = source_target_summary.pivot(index="source", columns="target", values="n_prioritized_interactions").fillna(0)
row_order = pivot_counts.sum(axis=1).sort_values(ascending=False).index
col_order = pivot_counts.sum(axis=0).sort_values(ascending=False).index
pivot_counts = pivot_counts.loc[row_order, col_order]

plt.figure(figsize=(max(10, 0.55 * pivot_counts.shape[1]), max(8, 0.45 * pivot_counts.shape[0])))
plt.imshow(pivot_counts.values, aspect="auto")
plt.colorbar(label="Number of prioritized LR interactions")
plt.xticks(range(pivot_counts.shape[1]), pivot_counts.columns, rotation=90, fontsize=8)
plt.yticks(range(pivot_counts.shape[0]), pivot_counts.index, fontsize=8)
plt.title("LIANA global main clean labels: prioritized LR interaction counts")
plt.xlabel("Target cell type")
plt.ylabel("Source cell type")
fig_path = savefig(FIG_DIR / f"{OUT_PREFIX}_source_target_prioritized_count_heatmap.png")
figure_records.append({"figure": str(fig_path), "description": "Clean-label heatmap of prioritized LIANA LR interaction counts by source-target pair."})

# Biological axis barplot
axis_plot = axis_summary.sort_values("n_prioritized_interactions", ascending=True)
plt.figure(figsize=(9, max(4, 0.45 * len(axis_plot))))
plt.barh(axis_plot["communication_axis"], axis_plot["n_prioritized_interactions"])
plt.xlabel("Unique prioritized LR pairs")
plt.ylabel("Communication axis")
plt.title("LIANA global main clean labels: communication axes")
fig_path = savefig(FIG_DIR / f"{OUT_PREFIX}_axis_summary_barplot.png")
figure_records.append({"figure": str(fig_path), "description": "Clean-label barplot of prioritized LR interaction counts across malignant/non-malignant communication axes."})

# Focused lineage barplot
bio_top = bio_summary.head(20).sort_values("n_interactions", ascending=True).copy()
bio_top["label"] = bio_top["source_lineage_simplified"] + " → " + bio_top["target_lineage_simplified"]
plt.figure(figsize=(9, max(5, 0.35 * len(bio_top))))
plt.barh(bio_top["label"], bio_top["n_interactions"])
plt.xlabel("Unique prioritized LR pairs")
plt.ylabel("Lineage axis")
plt.title("LIANA global main clean labels: top lineage-level axes")
fig_path = savefig(FIG_DIR / f"{OUT_PREFIX}_top_lineage_axis_barplot.png")
figure_records.append({"figure": str(fig_path), "description": "Clean-label barplot of top lineage-level LIANA axes."})

figure_manifest = pd.DataFrame(figure_records)
write_tsv_replace(figure_manifest, METADATA_DIR / f"{OUT_PREFIX}_figure_manifest.tsv")
display(figure_manifest)


In [ ]:
# Run summary and output manifest
summary = pd.DataFrame([
    {"item": "postprocessing_only", "value": True},
    {"item": "original_prefix", "value": OLD_PREFIX},
    {"item": "clean_prefix", "value": OUT_PREFIX},
    {"item": "n_prioritized_rows", "value": rank_sig_clean.shape[0]},
    {"item": "n_unique_lr_pairs", "value": rank_sig_clean["lr_pair"].nunique()},
    {"item": "n_source_target_pairs", "value": len(set(zip(rank_sig_clean["source"], rank_sig_clean["target"])))},
    {"item": "n_clean_celltypes", "value": len(set(rank_sig_clean["source"]).union(set(rank_sig_clean["target"])))},
    {"item": "completed_at", "value": datetime.now().isoformat(timespec="seconds")},
])
write_tsv_replace(summary, METADATA_DIR / f"{OUT_PREFIX}_run_summary.tsv")
display(summary)

manifest_paths = [
    METADATA_DIR / f"{OUT_PREFIX}_readiness.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_celltype_counts_used.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_rank_threshold_or_top_interactions.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_top_global_interactions.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_top_source_target_interactions.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_source_target_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_axis_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_biological_axis_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_ligand_receptor_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_malignant_to_microenvironment_interactions.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_microenvironment_to_malignant_interactions.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_malignant_state_to_malignant_state_interactions.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_focused_axis_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_figure_manifest.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_run_summary.tsv",
]
manifest = pd.DataFrame([
    {"path": str(p), "exists": p.exists(), "size_mb": round(p.stat().st_size / 1024 / 1024, 3) if p.exists() else np.nan}
    for p in manifest_paths
])
write_tsv_replace(manifest, METADATA_DIR / f"{OUT_PREFIX}_output_manifest.tsv")
display(manifest)


## Interpretation notes

- The original LIANA computation does **not** need to be rerun for this correction.
- The issue was only the naming of non-malignant high-confidence labels and downstream axis classification.
- Use the `_clean_labels` outputs for biological interpretation, figure captions and downstream validation.
- Keep the original all-results file for reproducibility; use clean-label summaries for thesis reporting.
